# WRN-28-10: measured 10-epoch Colab pilot
Run **10 epochs of SGD and 10 epochs of SAM**, seed 0, on the same GPU. Reuses the authors' original JAX/Flax `train_step`: CIFAR-10, batch 256, **eight logical sub-batches of 32**, Nesterov momentum 0.9, learning rate 0.1, weight decay 0.0005, SAM rho 0.05 and basic augmentation. The **200-epoch cosine schedule** stops after epoch 10; this is a timing/early-learning pilot, **not reproduction of the paper's final accuracy**. Each method processes 1,950 batches, unlike the previous ten-batch smoke test.

**Run:** upload to Colab → Runtime → Change runtime type → **L4 or A100 GPU** → set `RUN_EXPERIMENT = True` below → **Run all** → authorize Drive. No manual clone, pip install or kernel restart. A T4 previously ran out of memory on this unchanged workload; this notebook requires at least 20,000 MiB of GPU memory. It never reduces the model/batch or upgrades JAX automatically.

Default profile `colab_original` explicitly disables deterministic reductions, matching the earlier functional smoke settings. Option `js2_deterministic` enables the successful strict A100 preflight controls. Both methods share the chosen profile. This pilot verifies finite training/evaluation, parameter and BN updates, and checkpoint roundtrips; it does **not** rerun the main notebook's expensive numerical-equivalence diagnostics or replace its reproduction gate. Ten-epoch GPU execution is **(to verify)**.

Drive stores data, logs, measurements, two recent checkpoints per method and figures. Environment setup is reused within a surviving runtime; a **new Colab runtime must recreate the environment**, but reuses Drive's dataset cache. Allow several GB of Drive space. Keep Colab connected while running; a closed browser is not a guarantee the runtime survives. Run only one copy of this notebook at a time.

In [ ]:
RUN_EXPERIMENT = False  # Change ONLY this to True for the default run.
RUNTIME_PROFILE = "colab_original"  # or "js2_deterministic"; keep fixed during resume.
RESUME_RUN_ID = None  # After interruption: paste the printed run ID, then Run all.
MAX_SESSION_HOURS = 6.0  # Safety cap, NOT a prediction; includes setup and both methods.
KNOWN_PRIOR_GPU_HOURS = 1.974870124736945  # Last recorded cumulative JS2 ledger, including prior smoke.
TOTAL_BUDGET_HOURS = 40.0
# If more work occurred elsewhere, update KNOWN_PRIOR_GPU_HOURS first.
# The persistent pilot ledger adds ONLY attempts from this notebook to this baseline.

## Setup and execution
The isolated micromamba environment acts as this experiment's virtual environment: Python 3.8, JAX 0.2.6, CUDA jaxlib 0.1.57, Flax 0.2.2, CUDA 11.0 / cuDNN 8.0.5. Colab's own Python is unchanged. The clone uses `repro`, then checks out the recorded source revision. Only a runtime copy of requirements adjusts `absl-py` to 0.10.0 for pinned TensorFlow compatibility. Original source and requirements stay unchanged.

Every training step waits for GPU completion. CSVs separate input preparation, synchronized training, evaluation, checkpoint verification and Drive backup. The first five steps after each worker starts are marked **startup-affected**, not claimed as pure compiler time. GPU samples every two seconds report utilization, allocation and power; the sampled memory peak is not the true high-water mark or minimum memory required. Elapsed GPU hours are **not Colab compute units**. SAM's training loss is measured at perturbed weights; compare **clean test accuracy**, not the two training-loss values.

The six-hour worker deadline includes setup. It stops processes but **does not disconnect Colab**; disconnect/delete the runtime yourself after results are saved. A cumulative 40-hour ledger reserves this session's cap before work and settles actual time on handled exit. A crashed kernel leaves its reservation conservatively charged: do not delete/reset the ledger. Recorded time excludes allocation before execution/after completion, Drive authorization, report generation and unrecorded historical idle time. Drive copies/cleanup may extend beyond a worker deadline; this is not a provider billing cap.

In [ ]:
WORKER_SOURCE = r'''
import sys, json, time, hashlib, os, shutil, gc, csv
from pathlib import Path
from functools import partial
started = time.perf_counter()
import numpy as np
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
import jax, jax.numpy as jnp, flax
from flax import serialization
from absl import flags
from sam.sam_jax.models import load_model
from sam.sam_jax.datasets import dataset_source
from sam.sam_jax.training_utils import flax_training as official

cfg = json.loads(Path(sys.argv[1]).read_text())
method = sys.argv[2]
assert method in ('SGD', 'SAM')
out = Path(cfg['run']) / method
backup = Path(cfg['backup']) / method
out.mkdir(exist_ok=True); backup.mkdir(parents=True, exist_ok=True)
attempt = cfg['attempt']

def write_json(path, value):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False))
    os.replace(str(tmp), str(path))

def sync(tree):
    for x in jax.tree_leaves(tree):
        if hasattr(x, 'block_until_ready'): x.block_until_ready()
    return tree

def digest(tree, finite=False):
    h = hashlib.sha256()
    for x in jax.tree_leaves(tree):
        a = np.asarray(x)
        if finite: assert np.isfinite(a).all(), 'Non-finite parameter or state'
        h.update(str(a.shape).encode()); h.update(str(a.dtype).encode()); h.update(a.tobytes())
    return h.hexdigest()

def shard(batch):
    return {k: np.asarray(v).reshape((8, -1) + tuple(v.shape[1:])) for k, v in batch.items()}

def mirror_file(path):
    target = backup / path.name
    temp = target.with_name(target.name + '.tmp')
    shutil.copy2(str(path), str(temp)); os.replace(str(temp), str(target))

def checkpoint(epoch, optimizer, states, history):
    target = out / ('epoch_%03d' % epoch)
    temp = out / (target.name + '.partial')
    temp.mkdir(exist_ok=True)
    payload = dict(optimizer=optimizer, states=states)
    blob = serialization.to_bytes(payload)
    (temp / 'state.msgpack').write_bytes(blob)
    restored = serialization.from_bytes(payload, blob)
    for a, b in zip(jax.tree_leaves(payload), jax.tree_leaves(restored)):
        np.testing.assert_array_equal(np.asarray(a), np.asarray(b))
    meta = dict(epoch=epoch, step=int(optimizer.state.step), signature=cfg['signature'],
                sha256=hashlib.sha256(blob).hexdigest(), history=history, exact_roundtrip=True)
    write_json(temp / 'metadata.json', meta)
    del restored, payload, blob
    os.replace(str(temp), str(target))
    return target, meta

def backup_checkpoint(target):
    destination = backup / target.name
    temp = backup / (target.name + '.partial')
    if temp.exists(): shutil.rmtree(str(temp))
    shutil.copytree(str(target), str(temp))
    meta = json.loads((temp / 'metadata.json').read_text())
    assert hashlib.sha256((temp / 'state.msgpack').read_bytes()).hexdigest() == meta['sha256']
    os.replace(str(temp), str(destination))
    # Only committed checkpoints inside this run/method; retain two after verified backup.
    for root in (out, backup):
        committed = sorted(p for p in root.glob('epoch_[0-9][0-9][0-9]') if p.is_dir())
        for old in committed[:-2]: shutil.rmtree(str(old))

F = flags.FLAGS; F(['colab_10epoch'])
settings = dict(learning_rate=.1, weight_decay=.0005, gradient_clipping=5.0,
    use_rmsprop=False, sam_rho=.05 if method == 'SAM' else 0., sync_perturbations=False,
    label_smoothing=0., no_weight_decay_on_bn=False, ema_decay=0., use_test_set=True,
    use_additional_skip_connections=False, run_seed=0, compute_top_5_error_rate=False)
for key, value in settings.items(): setattr(F, key, value)
assert cfg['epochs'] == 10 and cfg['batch_size'] == 256 and cfg['seed'] == 0
assert len(jax.devices()) == 1 and jax.devices()[0].platform == 'gpu'
np.random.seed(0); tf.random.set_seed(0)
print(method, 'loading cached data and initializing original WRN-28-10', flush=True)
ds = dataset_source.Cifar10(256, 'basic', 'none')
# The original factory defaults to PRNGKey(0); no explicit key avoids its array-truthiness bug.
model, state = load_model.get_model('WideResnet28x10', 32, 32, 10)
sync((model, state))
parameter_count = sum(x.size for x in jax.tree_leaves(model.params))
initial_hash = digest(model.params)
optimizer = official.create_optimizer(model, 0.0)
states = sync(jax.tree_map(lambda x: jnp.stack([x] * 8), state))
del model, state
schedule = official.get_cosine_schedule(200, .1, 50000, 256)
# Original authors' update, including SAM, gradient averaging/clipping and clean-forward BN.
# One shared optimizer; independent BatchNorm states for the eight logical sub-batches.
step = jax.jit(jax.vmap(partial(official.train_step, learning_rate_fn=schedule, l2_reg=.0005),
    in_axes=(None, 0, 0, 0), out_axes=(None, 0, 0, None), axis_name='batch'))
evaluate = jax.jit(jax.vmap(official.eval_step, in_axes=(None, 0, 0), axis_name='batch'))
history = []; first_epoch = 1
committed = sorted(p for p in out.glob('epoch_[0-9][0-9][0-9]') if p.is_dir())
if committed:
    latest = committed[-1]; meta = json.loads((latest / 'metadata.json').read_text())
    assert meta['signature'] == cfg['signature'], 'Checkpoint/config mismatch'
    blob = (latest / 'state.msgpack').read_bytes()
    assert hashlib.sha256(blob).hexdigest() == meta['sha256'], 'Checkpoint damaged'
    restored = serialization.from_bytes(dict(optimizer=optimizer, states=states), blob)
    optimizer, states = sync((restored['optimizer'], restored['states']))
    assert int(optimizer.state.step) == meta['epoch'] * 195 == meta['step']
    history = meta['history']; first_epoch = meta['epoch'] + 1
    # Prefer finalized timing history when it matches the committed checkpoint.
    if (out / 'history.json').exists():
        finalized = json.loads((out / 'history.json').read_text())
        if len(finalized) == meta['epoch']: history = finalized
    del restored, blob
    print('Restored epoch', meta['epoch'], 'with optimizer and eight BN states.', flush=True)
identity = dict(method=method, parameter_count=int(parameter_count), initial_parameter_sha256=initial_hash,
    settings=settings, profile=cfg['profile'], signature=cfg['signature'],
    initialization_seconds=time.perf_counter()-started, resumed_from_epoch=first_epoch-1, attempt=attempt)
write_json(out / ('identity_' + attempt + '.json'), identity)
mirror_file(out / ('identity_' + attempt + '.json'))
print('Parameters:', parameter_count, '| initialization seconds:', round(identity['initialization_seconds'], 2), flush=True)

steps_path = out / ('steps_' + attempt + '.csv')
with steps_path.open('w', newline='') as sf:
    writer = csv.DictWriter(sf, fieldnames=['epoch', 'batch', 'step', 'input_seconds', 'train_seconds',
                                          'startup_affected', 'learning_rate', 'train_loss'])
    writer.writeheader(); total_steps_this_worker = 0
    for epoch in range(first_epoch, 11):
        epoch_start = time.perf_counter()
        previous_hash = digest(optimizer.target.params); bn_before = digest(states)
        times = []; losses = []; input_seconds = 0.
        train_start = time.perf_counter(); iterator = iter(ds.get_train(True)); count = 0
        while True:
            t = time.perf_counter()
            try: raw = next(iterator)
            except StopIteration: break
            batch = shard(raw)
            keys = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(0), int(optimizer.state.step)), 8)
            sync(keys); input_time = time.perf_counter() - t
            t = time.perf_counter()
            optimizer, states, metrics, lr = sync(step(optimizer, states, batch, keys))
            train_time = time.perf_counter() - t
            assert all(np.isfinite(np.asarray(v)).all() for v in metrics.values()), 'Non-finite training metric'
            loss = float(np.asarray(metrics['train_loss']).mean())
            total_steps_this_worker += 1; count += 1
            writer.writerow(dict(epoch=epoch, batch=count, step=int(optimizer.state.step),
                input_seconds=input_time, train_seconds=train_time,
                startup_affected=int(total_steps_this_worker <= 5), learning_rate=float(lr), train_loss=loss))
            sf.flush(); times.append(train_time); losses.append(loss); input_seconds += input_time
            if count == 1 or count % 25 == 0 or count == 195:
                print('%s epoch %d/10 batch %d/195 | synchronized step %.3fs | loss %.4f' %
                      (method, epoch, count, train_time, loss), flush=True)
        train_wall = time.perf_counter() - train_start
        assert count == 195 and int(optimizer.state.step) == epoch * 195
        del iterator, batch, raw, keys, metrics
        t = time.perf_counter(); sums = dict(error_rate=0., loss=0.); seen = 0
        for batch in ds.get_test():
            raw = {k: np.asarray(v) for k, v in batch.items()}; n = len(raw['label'])
            padded = ((n+7)//8)*8
            raw = {k: np.pad(v, [(0, padded-n)]+[(0, 0)]*(v.ndim-1)) for k, v in raw.items()}
            raw['mask'] = np.concatenate([np.ones(n), np.zeros(padded-n)]).astype(np.float32)
            metrics = sync(evaluate(optimizer.target, states, shard(raw)))
            for key in sums: sums[key] += float(np.asarray(metrics[key])[0])
            seen += n
        evaluation_seconds = time.perf_counter()-t
        assert seen == 10000 and all(np.isfinite(v) for v in sums.values())
        del batch, raw, metrics
        t = time.perf_counter()
        assert digest(optimizer.target.params, finite=True) != previous_hash, 'Parameters did not update'
        assert digest(states, finite=True) != bn_before, 'BatchNorm did not update'
        assert all(np.asarray(v).shape[0] == 8 for v in jax.tree_leaves(states))
        verification_seconds = time.perf_counter()-t
        row = dict(epoch=epoch, method=method, attempt=attempt, batches=count, train_examples=count*256,
            learning_rate=float(lr), train_loss=float(np.mean(losses)), train_wall_seconds=train_wall,
            input_seconds=input_seconds, synchronized_train_seconds=sum(times),
            median_step_seconds=float(np.median(times)), evaluation_seconds=evaluation_seconds,
            verification_seconds=verification_seconds, test_examples=seen,
            test_error_pct=100*sums['error_rate']/seen, test_accuracy_pct=100*(1-sums['error_rate']/seen),
            test_loss=sums['loss']/seen, parameters_updated=True, batchnorm_updated=True,
            startup_epoch=(epoch == first_epoch))
        history.append(row)
        t = time.perf_counter(); saved, meta = checkpoint(epoch, optimizer, states, history)
        row['checkpoint_seconds'] = time.perf_counter()-t; row['checkpoint_roundtrip'] = True
        t = time.perf_counter()
        meta['history'] = history; write_json(saved / 'metadata.json', meta)
        backup_checkpoint(saved); mirror_file(steps_path)
        row['backup_seconds'] = time.perf_counter()-t
        row['epoch_wall_seconds'] = time.perf_counter()-epoch_start
        write_json(out / 'history.json', history); mirror_file(out / 'history.json')
        print('%s epoch %d DONE | train %.1fs | eval %.1fs | clean test accuracy %.2f%%' %
              (method, epoch, train_wall, evaluation_seconds, row['test_accuracy_pct']), flush=True)
        gc.collect()
assert len(history) == 10
write_json(out / 'complete.json', dict(status='complete', method=method, epochs=10,
    signature=cfg['signature'], worker_wall_seconds=time.perf_counter()-started,
    final_test_accuracy_pct=history[-1]['test_accuracy_pct'], attempt=attempt))
mirror_file(out / 'complete.json')
print(method, 'TEN EPOCHS COMPLETE', flush=True)

'''

In [ ]:
import os, sys, json, time, hashlib, subprocess, shutil, tarfile, io, urllib.request
import threading, signal, uuid, csv, traceback, platform, fcntl
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path('/content/sam_colab_10epoch')
DRIVE = Path('/content/drive/MyDrive/sam_replication')
SOURCE_REVISION = 'f7b9ff46ee0deab8004d5508baf68b7e7c3dad1a'
REPO = ROOT / 'work' / 'sam'; PREFIX = ROOT / 'envs' / 'sam'; PYTHON = PREFIX / 'bin' / 'python'
SESSION = None

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False)); os.replace(str(tmp), str(path))

def command(args, log, environment=None, deadline=None, check=True):
    """Stream file logs; bound subprocesses even if the notebook kernel disappears."""
    args = list(map(str, args))
    if '-c' in args:
        compile(args[args.index('-c')+1], '<subprocess Python>', 'exec')
    log = Path(log); log.parent.mkdir(parents=True, exist_ok=True)
    remaining = int((deadline or (time.monotonic()+1800))-time.monotonic())
    if remaining <= 0: raise TimeoutError('Session deadline reached; no new process started')
    bounded = ['timeout', '--signal=INT', '--kill-after=15s', str(remaining)+'s']+args
    label = list(args[:5])
    if '-c' in label and label.index('-c')+1 < len(label):
        label[label.index('-c')+1] = '<embedded Python>'
    print('Running:', ' '.join(label), flush=True)
    with log.open('a') as output:
        output.write('\nCOMMAND '+repr(args)+'\n'); output.flush()
        offset = log.stat().st_size
        process = subprocess.Popen(bounded, stdout=output, stderr=subprocess.STDOUT,
                                   env=environment, start_new_session=True)
        try:
            with log.open() as reader:
                reader.seek(offset); heartbeat = time.monotonic()
                while process.poll() is None:
                    chunk = reader.read()
                    if chunk: print(chunk, end='', flush=True)
                    if time.monotonic()-heartbeat > 30:
                        print('[still running; log:', log.name+']', flush=True); heartbeat = time.monotonic()
                    time.sleep(1)
                print(reader.read(), end='', flush=True)
            if check and process.returncode: raise RuntimeError('Exit %s; see %s' % (process.returncode, log))
            return process.returncode
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try: process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL); process.wait()

def environment():
    env = os.environ.copy()
    env.update(PATH=str(PREFIX/'bin')+':'+env.get('PATH', ''),
        LD_LIBRARY_PATH=str(PREFIX/'lib')+':'+env.get('LD_LIBRARY_PATH', ''),
        PYTHONPATH=str(REPO.parent), CUDA_HOME=str(PREFIX), CUDA_PATH=str(PREFIX),
        CUDA_VISIBLE_DEVICES='0', XLA_PYTHON_CLIENT_PREALLOCATE='false', PYTHONHASHSEED='0',
        TFDS_DATA_DIR=str(ROOT/'tensorflow_datasets'), TF_DETERMINISTIC_OPS='0',
        TF_CUDNN_DETERMINISTIC='1' if RUNTIME_PROFILE == 'js2_deterministic' else '0',
        XLA_FLAGS='--xla_gpu_cuda_data_dir='+str(ROOT/'cuda')+
            ' --xla_gpu_deterministic_reductions='+('true' if RUNTIME_PROFILE == 'js2_deterministic' else 'false'))
    return env

VERSION_CHECK = '''
import sys
from importlib.metadata import version
assert sys.version_info[:2] == (3,8)
expected={'jax':'0.2.6','jaxlib':'0.1.57+cuda110','flax':'0.2.2','numpy':'1.18.5','tensorflow':'2.3.1'}
for package,want in expected.items():
    got=version(package); assert got==want,(package,got,want)
import numpy, tensorflow, flax, jax
print('Verified pinned packages:',expected,flush=True)
'''
GPU_CHECK = VERSION_CHECK + '''
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
assert len(jax.devices())==1 and jax.devices()[0].platform=='gpu'
x=jax.jit(lambda a:a@a)(jax.numpy.ones((2,2))); x.block_until_ready()
print('GPU initialization PASS:',jax.devices(),flush=True)
'''

def pinned_cuda_ready():
    packages = {}
    for path in (PREFIX/'conda-meta').glob('*.json'):
        record = json.loads(path.read_text())
        packages[record['name']] = record['version']
    return (packages.get('cudatoolkit', '').startswith('11.0.') and
            packages.get('cudnn', '').split('.')[:3] == ['8', '0', '5'] and packages.get('cuda-nvcc') == '11.3.122')

def setup(deadline):
    if not REPO.exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        command(['git', 'clone', '--branch', 'repro', 'https://github.com/Siddharth-magesh/sam-replication.git', REPO], SESSION/'setup.log', deadline=deadline)
        command(['git', '-C', REPO, 'checkout', '--detach', SOURCE_REVISION], SESSION/'setup.log', deadline=deadline)
    actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
    assert actual == SOURCE_REVISION, 'Existing runtime checkout differs; use a fresh ROOT.'
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=no'], text=True, timeout=30).strip(), 'Tracked source modifications'
    ready = False
    if PYTHON.exists() and (PREFIX/'bin'/'ptxas').exists() and pinned_cuda_ready():
        ready = command([PYTHON, '-c', VERSION_CHECK], SESSION/'environment_check.log', environment(), deadline, check=False) == 0
    if not ready:
        manager = ROOT/'bin'/'micromamba'
        if not manager.exists():
            with urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest', timeout=120) as response:
                data = response.read()
            with tarfile.open(fileobj=io.BytesIO(data), mode='r:bz2') as archive:
                member = archive.getmember('bin/micromamba'); assert member.isfile()
                manager.parent.mkdir(parents=True, exist_ok=True)
                manager.write_bytes(archive.extractfile(member).read())
            manager.chmod(0o755)
        os.environ['MAMBA_ROOT_PREFIX'] = str(ROOT/'mamba')
        action = 'install' if PYTHON.exists() else 'create'
        command([manager, action, '-p', PREFIX, '-c', 'conda-forge', '--override-channels',
                 'python=3.8', 'pip=24.0', 'cudatoolkit=11.0', 'cudnn=8.0.5', '-y'], SESSION/'setup.log', deadline=deadline)
        command([manager, 'install', '-p', PREFIX, '-c', 'nvidia/label/cuda-11.3.1', '--override-channels',
                 'cuda-nvcc=11.3.122', '-y'], SESSION/'setup.log', deadline=deadline)
        lines = (REPO/'requirements.txt').read_text().splitlines()
        requirements = ROOT/'requirements_runtime.txt'
        requirements.write_text('\n'.join(s.replace('absl-py==0.11.0', 'absl-py==0.10.0')
            for s in lines if not s.startswith('jaxlib'))+'\n')
        command([PYTHON, '-m', 'pip', 'install', 'jaxlib==0.1.57+cuda110', '-f',
                 'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'], SESSION/'setup.log', deadline=deadline)
        command([PYTHON, '-m', 'pip', 'install', '-r', requirements], SESSION/'setup.log', deadline=deadline)
    assert pinned_cuda_ready(), 'CUDA/cuDNN pins differ; inspect setup log instead of upgrading JAX'
    atomic_json(SESSION/'cuda_packages.json', [json.loads(p.read_text()) for p in (PREFIX/'conda-meta').glob('*.json')
        if p.name.startswith(('cudatoolkit-', 'cudnn-', 'cuda-nvcc-'))])
    candidates = sorted(PREFIX.rglob('libdevice*.bc')); assert candidates, 'CUDA libdevice missing'
    libdir = ROOT/'cuda'/'nvvm'/'libdevice'; libdir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(candidates[0]), str(libdir/'libdevice.10.bc'))
    command([PREFIX/'bin'/'ptxas', '--version'], SESSION/'environment_check.log', environment(), deadline)
    command([PYTHON, '-c', GPU_CHECK], SESSION/'environment_check.log', environment(), deadline)
    command([PYTHON, '-m', 'pip', 'freeze'], SESSION/'package_versions.txt', environment(), deadline)

DATA_SETUP = '''
import os
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
import tensorflow_datasets as tfds
builder=tfds.builder('cifar10',data_dir=os.environ['TFDS_DATA_DIR'])
builder.download_and_prepare()
assert builder.info.splits['train'].num_examples==50000
assert builder.info.splits['test'].num_examples==10000
print(builder.info,flush=True)
'''

def gpu_sampler(path, stop):
    fields = ['timestamp', 'name', 'utilization.gpu', 'memory.used', 'memory.total', 'power.draw']
    with Path(path).open('w', newline='') as output:
        writer = csv.writer(output); writer.writerow(['elapsed_seconds']+fields)
        start = time.monotonic()
        while not stop.is_set():
            try:
                value = subprocess.check_output(['nvidia-smi', '-i', '0', '--query-gpu='+','.join(fields),
                    '--format=csv,noheader,nounits'], text=True, timeout=8).strip()
                writer.writerow([time.monotonic()-start]+next(csv.reader([value]))); output.flush()
            except Exception as exc: print('GPU sampler:', str(exc), flush=True)
            stop.wait(2)

def execute_pilot():
    global SESSION
    if not RUN_EXPERIMENT:
        print('Disabled. Choose L4/A100 GPU, set RUN_EXPERIMENT=True, then Run all.'); return
    assert sys.platform == 'linux' and platform.machine() == 'x86_64', 'Use Colab Linux GPU runtime'
    assert RUNTIME_PROFILE in ('colab_original', 'js2_deterministic')
    assert 0 < MAX_SESSION_HOURS <= 6 and TOTAL_BUDGET_HOURS == 40
    assert 0 <= KNOWN_PRIOR_GPU_HOURS < 40
    # Validate embedded Python BEFORE downloads, environment repair or budget reservation.
    for label, source in [('worker', WORKER_SOURCE), ('versions', VERSION_CHECK),
                          ('GPU initialization', GPU_CHECK), ('dataset', DATA_SETUP)]:
        compile(source, '<'+label+'>', 'exec')
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT.mkdir(parents=True, exist_ok=True)
    with (ROOT/'pilot.lock').open('w') as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        return locked_pilot()

def locked_pilot():
    global SESSION
    gpu = subprocess.check_output(['nvidia-smi', '-i', '0', '--query-gpu=name,memory.total',
                                  '--format=csv,noheader,nounits'], text=True, timeout=15).strip()
    print('Allocated GPU:', gpu)
    assert float(gpu.rsplit(',', 1)[1]) >= 20000, 'Select L4/A100; unchanged SAM workload exhausted a T4.'
    run_id = RESUME_RUN_ID or (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
    assert Path(run_id).name == run_id and run_id not in ('.', '..'), 'Invalid run ID'
    persistent = DRIVE/'colab_10epoch'; persistent.mkdir(parents=True, exist_ok=True)
    backup = persistent/run_id; SESSION = ROOT/'runs'/run_id
    if RESUME_RUN_ID:
        assert backup.is_dir(), 'Resume run missing on Drive'
        if not SESSION.exists(): shutil.copytree(str(backup), str(SESSION))
    else:
        assert not backup.exists() and not SESSION.exists()
        SESSION.mkdir(parents=True); backup.mkdir()
    config = dict(epochs=10, batch_size=256, seed=0, logical_replicas=8, schedule_epochs=200, gpu=gpu,
        profile=RUNTIME_PROFILE, source_revision=SOURCE_REVISION,
        worker_sha256=hashlib.sha256(WORKER_SOURCE.encode()).hexdigest())
    signature = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    if RESUME_RUN_ID:
        assert json.loads((SESSION/'config.json').read_text())['signature'] == signature, 'Resume config/code mismatch'
    attempt = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]
    config.update(signature=signature, run=str(SESSION), backup=str(backup), attempt=attempt, gpu=gpu)
    ledger_path = persistent/'pilot_gpu_budget.json'
    ledger = json.loads(ledger_path.read_text()) if ledger_path.exists() else dict(prior_hours=KNOWN_PRIOR_GPU_HOURS, attempts=[])
    ledger['prior_hours'] = max(ledger['prior_hours'], KNOWN_PRIOR_GPU_HOURS)
    charged = ledger['prior_hours']+sum(a['charged_hours'] for a in ledger['attempts'])
    assert charged+MAX_SESSION_HOURS <= 40, 'Cannot reserve session within cumulative budget. Inspect ledger; do not reset.'
    entry = dict(attempt=attempt, run_id=run_id, status='reserved', charged_hours=MAX_SESSION_HOURS)
    ledger['attempts'].append(entry); atomic_json(ledger_path, ledger)
    started = time.monotonic(); deadline = started+MAX_SESSION_HOURS*3600-30
    timings = {}; status = 'failed'
    print('RUN ID:', run_id, '| Drive:', backup, '| reserved hours:', MAX_SESSION_HOURS, flush=True)
    atomic_json(SESSION/'config.json', config); (SESSION/'worker.py').write_text(WORKER_SOURCE)
    # Save provenance BEFORE work; recovery remains possible after abrupt kernel loss.
    shutil.copy2(str(SESSION/'config.json'), str(backup/'config.json'))
    shutil.copy2(str(SESSION/'worker.py'), str(backup/'worker.py'))
    try:
        t = time.monotonic()
        try:
            setup(deadline)
        finally:
            timings['environment_setup_seconds'] = time.monotonic()-t
        atomic_json(SESSION/'runtime_environment.json', {k: v for k, v in environment().items() if k in (
            'XLA_FLAGS', 'TF_CUDNN_DETERMINISTIC', 'TF_DETERMINISTIC_OPS', 'CUDA_VISIBLE_DEVICES',
            'XLA_PYTHON_CLIENT_PREALLOCATE', 'TFDS_DATA_DIR', 'PYTHONHASHSEED')})
        t = time.monotonic()
        cache = DRIVE/'tensorflow_datasets'; local = ROOT/'tensorflow_datasets'
        if (cache/'cifar10').exists() and not (local/'cifar10').exists():
            print('Copying cached CIFAR-10 from Drive.', flush=True)
            shutil.copytree(str(cache/'cifar10'), str(local/'cifar10'))
        command([PYTHON, '-c', DATA_SETUP], SESSION/'dataset.log', environment(), deadline)
        if not (cache/'cifar10').exists():
            cache.mkdir(parents=True, exist_ok=True)
            temp = cache/('cifar10.partial_'+attempt)
            shutil.copytree(str(local/'cifar10'), str(temp)); os.replace(str(temp), str(cache/'cifar10'))
        timings['data_prepare_and_cache_seconds'] = time.monotonic()-t
        for method in ('SGD', 'SAM'):
            complete = SESSION/method/'complete.json'
            if complete.exists():
                saved = json.loads(complete.read_text()); assert saved['signature'] == signature and saved['epochs'] == 10
                print(method, 'already complete; skipping.'); continue
            t = time.monotonic(); stop = threading.Event()
            sampler = threading.Thread(target=gpu_sampler,
                args=(SESSION/('gpu_'+method+'_'+attempt+'.csv'), stop), daemon=True)
            sampler.start()
            try:
                command([PYTHON, '-u', SESSION/'worker.py', SESSION/'config.json', method],
                        SESSION/(method+'_'+attempt+'.log'), environment(), deadline)
            finally:
                stop.set(); sampler.join(timeout=10)
                timings[method+'_worker_seconds'] = time.monotonic()-t
        identities = [json.loads(sorted((SESSION/method).glob('identity_*.json'))[-1].read_text()) for method in ('SGD', 'SAM')]
        assert identities[0]['initial_parameter_sha256'] == identities[1]['initial_parameter_sha256']
        assert identities[0]['parameter_count'] == identities[1]['parameter_count']
        status = 'complete'; print('BOTH TEN-EPOCH RUNS COMPLETE', flush=True)
    except BaseException:
        (SESSION/('failure_'+attempt+'.txt')).write_text(traceback.format_exc())
        print(traceback.format_exc(), flush=True)
        print('Stopped; completed epochs preserved. Results below may be partial.', flush=True)
    finally:
        receipt = dict(status=status, run_id=run_id, attempt=attempt, timings=timings,
            measured_elapsed_seconds=time.monotonic()-started, profile=RUNTIME_PROFILE, gpu=gpu)
        atomic_json(SESSION/('attempt_'+attempt+'.json'), receipt)
        try:
            for path in SESSION.iterdir():
                if path.is_file(): shutil.copy2(str(path), str(backup/path.name))
            # Preserve partial-step CSVs too, even if the worker failed before its next epoch backup.
            for method in ('SGD', 'SAM'):
                method_out = SESSION/method; method_backup = backup/method
                if method_out.is_dir():
                    method_backup.mkdir(exist_ok=True)
                    for path in method_out.iterdir():
                        if path.is_file(): shutil.copy2(str(path), str(method_backup/path.name))
        finally:
            entry.update(status=status, charged_hours=(time.monotonic()-started)/3600)
            atomic_json(ledger_path, ledger); atomic_json(SESSION/'budget_snapshot.json', ledger)
        print('Recorded attempt hours:', round(entry['charged_hours'], 4), '| cumulative:',
              round(ledger['prior_hours']+sum(a['charged_hours'] for a in ledger['attempts']), 4), '/40')
        print('Results:', SESSION, '\nDrive:', backup, '\nDisconnect/delete runtime when finished.')

execute_pilot()

## Measured results and recovery
Read actual worker/epoch times first. Plots show clean test accuracy, training/evaluation time, step-time distributions and live GPU memory. The secondary 200-epoch extrapolation uses the median of completed **non-startup epochs**. It is an estimate, not permission to launch a full run. One seed at epoch 10 does not establish reproduction of the paper's final result.

On interruption, set `RESUME_RUN_ID` to the printed ID, keep the profile unchanged, and Run all. Resume restores the last committed optimizer and all eight BN states, skips completed methods and logs a new attempt. TF shuffle/augmentation state is not checkpointed, so resumed trajectories are not guaranteed identical to uninterrupted ones. Startup epochs of resumed workers are excluded from steady estimates. Partial steps remain in per-attempt CSVs; failed attempts remain in the ledger. If a cell fails, manually run the results cell to inspect available measurements.

Artifacts: `MyDrive/sam_replication/colab_10epoch/<RUN_ID>/`. Dataset cache: `MyDrive/sam_replication/tensorflow_datasets/cifar10/`. Download the **executed** notebook via File → Download → Download .ipynb to keep displayed outputs. Do not commit raw logs, datasets or checkpoints. The main reproduction notebook, its A100 requirement, full seed matrix and previous executed notebooks remain unchanged.

In [ ]:
REPORT_SOURCE = r'''
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
root = Path(sys.argv[1]); rows = []; summaries = []
for method in ('SGD', 'SAM'):
    path = root/method/'history.json'
    if not path.exists(): continue
    history = json.loads(path.read_text()); rows.extend(history)
    df = pd.DataFrame(history)
    if 'epoch_wall_seconds' not in df: df['epoch_wall_seconds'] = np.nan
    steady = df[~df.startup_epoch].dropna(subset=['epoch_wall_seconds'])
    summaries.append(dict(method=method, completed_epochs=len(df),
        measured_epoch_minutes=float(df.epoch_wall_seconds.sum(min_count=1))/60,
        final_clean_test_accuracy_pct=float(df.test_accuracy_pct.iloc[-1]),
        median_steady_epoch_seconds=float(steady.epoch_wall_seconds.median()) if len(steady) else None,
        estimated_200epoch_hours=(float(df.epoch_wall_seconds.iloc[0])+199*float(steady.epoch_wall_seconds.median()))/3600 if len(steady) else None))
if not rows:
    print('No completed epochs yet. Inspect setup/dataset/worker logs and failure file.'); sys.exit(0)
df = pd.DataFrame(rows); df.to_csv(root/'epoch_summary.csv', index=False)
summary = pd.DataFrame(summaries); summary.to_csv(root/'timing_summary.csv', index=False)
print('\nCOMPLETED EPOCHS: checkpoint/backup included; setup excluded. Missing timings remain blank.\n'+summary.to_string(index=False))
print('\n200-epoch column is an extrapolation, NOT a measured result; no full run launched.')
receipts = [json.loads(p.read_text()) for p in sorted(root.glob('attempt_*.json'))]
print('\nATTEMPT WALL TIME including setup, data, failed/partial work:')
for r in receipts:
    print(r['attempt'], r['status'], round(r['measured_elapsed_seconds']/60, 2), 'minutes', r['timings'])
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for method in ('SGD', 'SAM'):
    part = df[df.method == method]
    if len(part) == 0: continue
    axes[0, 0].plot(part.epoch, part.test_accuracy_pct, 'o-', label=method)
    axes[0, 1].plot(part.epoch, part.train_wall_seconds/60, 'o-', label=method+' training')
    axes[0, 1].plot(part.epoch, part.evaluation_seconds/60, '--', label=method+' evaluation')
    frames = [pd.read_csv(p) for p in sorted((root/method).glob('steps_*.csv'))]
    frames = [p for p in frames if len(p)]
    if frames:
        steps = pd.concat(frames, ignore_index=True)
        stable = steps[steps.startup_affected == 0].train_seconds
        if len(stable):
            axes[1, 0].hist(stable, bins=50, alpha=.5, label=method)
            print(method, 'synchronized step seconds excluding first five per worker:',
                'median=', round(float(stable.median()), 4), 'p90=', round(float(stable.quantile(.9)), 4),
                '| includes interrupted attempts, if any; see CSVs')
    for p in sorted(root.glob('gpu_'+method+'_*.csv')):
        g = pd.read_csv(p)
        if len(g):
            memory = pd.to_numeric(g['memory.used'], errors='coerce')/1024
            axes[1, 1].plot(g.elapsed_seconds/60, memory, label=method+' '+p.stem[-6:])
            utilization = pd.to_numeric(g['utilization.gpu'], errors='coerce')
            print(method, 'sampled peak GiB:', round(float(memory.max()), 2),
                '| mean worker utilization:', round(float(utilization.mean()), 1), '%')
axes[0, 0].set(xlabel='Completed epoch', ylabel='Clean test accuracy (%)', title='Full 10,000-example test set')
axes[0, 1].set(xlabel='Completed epoch', ylabel='Actual minutes', title='First epoch includes startup/compilation effects')
axes[1, 0].set(xlabel='Synchronized training step seconds', ylabel='Batches', title='Measured steps (startup five excluded)')
axes[1, 1].set(xlabel='Minutes since worker start', ylabel='GPU allocation (GiB)', title='Sampled memory, not minimum requirement')
for ax in axes.flat:
    ax.grid(alpha=.2)
    if ax.get_legend_handles_labels()[0]: ax.legend()
fig.suptitle('WRN-28-10 / CIFAR-10 | '+json.loads((root/'config.json').read_text())['profile'])
fig.tight_layout(); fig.savefig(root/'measured_results.png', dpi=160); plt.close(fig)

'''

if RUN_EXPERIMENT and SESSION is not None and PYTHON.exists() and (SESSION/'config.json').exists():
    from IPython.display import display, Image
    command([PYTHON, '-c', REPORT_SOURCE, SESSION], SESSION/'report.log', environment(), time.monotonic()+180)
    backup = Path(json.loads((SESSION/'config.json').read_text())['backup'])
    for name in ('epoch_summary.csv', 'timing_summary.csv', 'measured_results.png', 'report.log', 'budget_snapshot.json'):
        path = SESSION/name
        if path.exists(): shutil.copy2(str(path), str(backup/name))
    if (SESSION/'measured_results.png').exists():
        display(Image(filename=str(SESSION/'measured_results.png')))
        print('Saved figures and tables:', backup)
    else:
        print('No completed-epoch figures available. Setup/failure logs saved:', backup)
else:
    print('No pilot launched. Set RUN_EXPERIMENT=True and Run all on an L4/A100 Colab GPU.')